In [1]:
import pandas as pd
import numpy as np

In [3]:
ABtest = pd.read_csv("ab_data.csv")
ABtest.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [4]:
ABtest.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [5]:
ABtest.isnull().sum()

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

In [6]:
# Check the grouping error
# Correct: Treatment group - New page; Control group - Old page
t_old_error = len(ABtest[(ABtest['group']=='treatment') & (ABtest['landing_page']=='old_page')])
c_new_error = len(ABtest[(ABtest['group']=='control') & (ABtest['landing_page']=='new_page')])

print('Treatment group with old page: {}; Control group with new page: {}'.format(t_old_error, c_new_error))

Treatment group with old page: 1965; Control group with new page: 1928


In [8]:
# Leave the correct grouping
ABtest = ABtest[((ABtest['group']=='treatment') & (ABtest['landing_page']=='new_page')) |
                ((ABtest['group']=='control') & (ABtest['landing_page']=='old_page'))]
print('The correct grouping has', len(ABtest))

The correct grouping has 290585


In [13]:
duplicates = ABtest[ABtest.duplicated(subset=['user_id'], keep=False)]
duplicates

,user_id,timestamp,group,landing_page,converted
1899,773192,2017-01-09 05:37:58.781806,treatment,new_page,0
2893,773192,2017-01-14 02:55:59.590927,treatment,new_page,0


In [14]:
ABtest.drop_duplicates(subset=['user_id'], inplace=True, keep='first')

In [15]:
ABtest.info()

<class 'pandas.core.frame.DataFrame'>
Index: 290584 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       290584 non-null  int64 
 1   timestamp     290584 non-null  object
 2   group         290584 non-null  object
 3   landing_page  290584 non-null  object
 4   converted     290584 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 13.3+ MB


In [16]:
ABtest.to_csv('cleaned_ab_data.csv', index=False)

Statistic

In [18]:
# Check the distribution
New = ABtest[ABtest.landing_page=='new_page'].shape[0] / ABtest.shape[0]
print('New page user:', '%.2f%%'%(New*100))

Old = ABtest[ABtest.landing_page=='old_page'].shape[0] / ABtest.shape[0]
print('Old page user:', '%.2f%%'%(Old*100))


New page user: 50.01%
Old page user: 49.99%


In [ ]:
# Z-test

# Number of user in each group
n_old = ABtest.query('group=="control"').shape[0]
n_new = ABtest.query('group=="treatment"').shape[0]

# Number of converted user
convert_old = ABtest.query('group=="control" & converted==1').shape[0]
convert_new = ABtest.query('group=="treatment" & converted==1').shape[0]

# Converted rate
p_old = convert_old/n_old
p_new = convert_new/n_new

In [27]:
from statsmodels.stats.proportion import proportions_ztest

count = [convert_old, convert_new]
nobs = [n_old, n_new]

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs
)

In [28]:
from scipy.stats import norm

# Critical value at α = 0.05 (one-tailed test)
z_alpha = norm.ppf(0.05)

print(
    f"Control Group:\n"
    f"Number of Users: {n_old}\n"
    f"Converted Users: {convert_old}\n"
    f"Conversion Rate: {p_old:.4f}\n"
)

print(
    f"Treatment Group:\n"
    f"Number of Users: {n_new}\n"
    f"Converted Users: {convert_new}\n"
    f"Conversion Rate: {p_new:.4f}\n"
)

print(f"Z-test: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Critical Value: {z_alpha:.4f}")

result = (
    "Reject the Null Hypothesis"
    if abs(z_stat) > abs(z_alpha)
    else "Fail to Reject the Null Hypothesis"
)

print(result)

Control Group:
Number of Users: 145274
Converted Users: 17489
Conversion Rate: 0.1204

Treatment Group:
Number of Users: 145310
Converted Users: 17872
Conversion Rate: 0.1230

Z-test: -2.1484
P-value: 0.0317
Critical Value: -1.6449
Reject the Null Hypothesis


In [ ]:
# Pooled standard deviation
std_old = ABtest[ABtest.landing_page == "old_page"].converted.std()
std_new = ABtest[ABtest.landing_page == "new_page"].converted.std()

pooled_std = np.sqrt(
    ((n_old - 1) * std_old**2 + (n_new - 1) * std_new**2)
    / (n_old + n_new - 2)
)

# Cohen's d effect size
cohen_d = (p_old - p_new) / pooled_std

print(f"Cohen's d Effect Size: {cohen_d:.4f}")

Cohen's d Effect Size: -0.0080
